# 02 - Nettoyage & Préparation

Objectif : nettoyer chaque dataset, normaliser les codes INSEE / département, et créer des DataFrames prêts à l'analyse.

In [1]:
import sys
import zipfile
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import DATA_RAW, DATA_PROCESSED, DATA_EXTERNAL
from src.clean_data import load_csv, normalize_code_insee, normalize_code_departement, clean_pct, save_processed

## 2.1 Communes SRU

In [2]:
sru = load_csv(DATA_RAW / "donnees-sru-data-gouv-2025-v2.csv", sep=";")
print(f"SRU: {sru.shape}")
sru.columns.tolist()

SRU: (2196, 23)


['hexagone_drom',
 'Region',
 'Departement',
 'Code_Departement',
 'Code_INSEE_commune',
 'Nom_commune',
 'Population_municipale_01_01_2025',
 'Code_SIREN_EPCI',
 'Nom_EPCI',
 'EPCI_SRU',
 'Code_unité_urbaine',
 'Nom_unité_urbaine',
 'UU_SRU',
 'Commune_isolée_article_L_302_5_CCH',
 'Commune_sru_au_01_01_2025',
 'Commune_sru_au_01_01_2024',
 'Nombre_lls_ Inventaire_au_01_01_2024',
 'Taux_SRU_au_01_01_2024',
 'commune_deficitaire',
 'Commune_carencée',
 'Commune_exemptée_2023_2025',
 'Taux_cible_commune_2023_2025',
 'Prélèvement_net_2025_dont_majoration']

In [3]:
sru["code_insee"] = sru["Code_INSEE_commune"].apply(normalize_code_insee)
sru["code_departement"] = sru["Code_Departement"].apply(normalize_code_departement)

for col in ["Population_municipale_01_01_2025", "Nombre_lls_ Inventaire_au_01_01_2024"]:
    sru[col] = pd.to_numeric(sru[col], errors="coerce")

for col in ["Taux_SRU_au_01_01_2024", "Taux_cible_commune_2023_2025"]:
    sru[col] = clean_pct(sru[col])

for col in ["Commune_sru_au_01_01_2025", "commune_deficitaire", "Commune_carencée"]:
    sru[col] = sru[col].map({"oui": True, "non": False, 1: True, 0: False, "1": True, "0": False})

sru_clean = sru[[
    "code_insee", "code_departement", "Nom_commune",
    "Population_municipale_01_01_2025", "Code_SIREN_EPCI", "Nom_EPCI",
    "Commune_sru_au_01_01_2025", "commune_deficitaire", "Commune_carencée",
    "Nombre_lls_ Inventaire_au_01_01_2024", "Taux_SRU_au_01_01_2024",
    "Taux_cible_commune_2023_2025", "Prélèvement_net_2025_dont_majoration",
]].copy()

print(f"SRU nettoyé: {sru_clean.shape}")
print(f"  Communes SRU: {sru_clean['Commune_sru_au_01_01_2025'].sum():,}")
print(f"  Déficitaires: {sru_clean['commune_deficitaire'].sum():,}")
print(f"  Carencées: {sru_clean['Commune_carencée'].sum():,}")
save_processed(sru_clean, "sru")

SRU nettoyé: (2196, 13)
  Communes SRU: 2,196
  Déficitaires: 1,276
  Carencées: 335
  -> C:\Users\Salty\Downloads\defi-municipales-2026-enjeux\data\processed\sru.parquet (2,196 rows, 13 cols)


WindowsPath('C:/Users/Salty/Downloads/defi-municipales-2026-enjeux/data/processed/sru.parquet')

## 2.2 Carte des loyers 2025

In [4]:
loyer_files = {
    "appartement": "pred-app-mef-dhup.csv",
    "t1t2": "pred-app12-mef-dhup.csv",
    "t3plus": "pred-app3-mef-dhup.csv",
    "maison": "pred-mai-mef-dhup.csv",
}

loyers_all = []
for label, filename in loyer_files.items():
    path = DATA_RAW / filename
    if not path.exists():
        print(f"  {label}: MANQUANT")
        continue
    df = load_csv(path, sep=";")
    df["type_logement"] = label
    df["code_insee"] = df["INSEE_C"].apply(normalize_code_insee)
    df["code_departement"] = df["DEP"].apply(normalize_code_departement)
    df["loypredm2"] = df["loypredm2"].astype(str).str.replace(",", ".", regex=False).pipe(pd.to_numeric, errors="coerce")
    loyers_all.append(df[["code_insee", "code_departement", "LIBGEO", "type_logement", "loypredm2", "nbobs_com"]])
    print(f"  {label}: {df.shape[0]:,} communes")

loyers = pd.concat(loyers_all, ignore_index=True)
print(f"\nLoyers total: {loyers.shape}")
save_processed(loyers, "loyers")

  appartement: 34,900 communes


  t1t2: 34,900 communes


  t3plus: 34,900 communes


  maison: 34,900 communes

Loyers total: (139600, 6)
  -> C:\Users\Salty\Downloads\defi-municipales-2026-enjeux\data\processed\loyers.parquet (139,600 rows, 6 cols)


WindowsPath('C:/Users/Salty/Downloads/defi-municipales-2026-enjeux/data/processed/loyers.parquet')

## 2.3 Éducation prioritaire

In [5]:
ep = load_csv(DATA_RAW / "fr-en-etablissements-ep.csv", sep=";")
print(f"Éducation prioritaire: {ep.shape}")
print(f"  Colonnes: {ep.columns.tolist()}")

Éducation prioritaire: (8501, 24)


  Colonnes: ['UAI', 'EP 2022-2023', 'Nom', 'Type', 'Statut', 'Académie', 'Département', 'Commune', 'Région', 'UAI tête de réseau', 'QP à proximité O N', 'QP à proximité', 'Nom du QP', "Nombre d'élèves", 'Code_postal', 'Code commune', 'Code département', 'Code académie', 'Code région', "Nature de l'établissement", 'Code nature', 'position', 'latitude', 'longitude']


In [6]:
ep["code_departement"] = ep["Code département"].apply(normalize_code_departement)

ep_clean = ep[[
    "UAI", "Nom", "Type", "Statut", "Académie", "Département",
    "code_departement", "Commune", "Code commune", "Code_postal",
    "EP 2022-2023", "Nombre d'élèves",
]].copy()

ep_clean.rename(columns={"EP 2022-2023": "ep_2022"}, inplace=True)

print(f"\nÉtablissements EP nettoyé: {ep_clean.shape}")
print(ep_clean["ep_2022"].value_counts())
save_processed(ep_clean, "education_prioritaire")


Établisements EP nettoyé: (8501, 12)
ep_2022
REP        4851
REP+       2796
HORS EP     854
Name: count, dtype: int64
  -> C:\Users\Salty\Downloads\defi-municipales-2026-enjeux\data\processed\education_prioritaire.parquet (8,501 rows, 12 cols)


WindowsPath('C:/Users/Salty/Downloads/defi-municipales-2026-enjeux/data/processed/education_prioritaire.parquet')

## 2.4 Effectifs élèves par école

In [7]:
eleves = load_csv(DATA_RAW / "fr-en-ecoles-effectifs-nb_classes.csv", sep=";")
print(f"Effectifs élèves: {eleves.shape}")
print(f"  Colonnes: {eleves.columns.tolist()}")

Effectifs élèves: (809225, 28)
  Colonnes: ['Rentrée scolaire', 'Code région académique', 'Code région Insee', 'Région académique', 'Code académie', 'Académie', 'Code département', 'Département', 'Code Postal', 'Commune', "Numéro de l'école", 'Dénomination principale', 'Patronyme', 'Secteur', 'REP', 'REP +', 'Nombre total de classes', "Nombre total d'élèves", "Nombre d'élèves en pré-élémentaire hors ULIS", "Nombre d'élèves en élémentaire hors ULIS", "Nombre d'élèves en ULIS", "Nombre d'élèves en UEEA", "Nombre d'élèves en CP hors ULIS", "Nombre d'élèves en CE1 hors ULIS", "Nombre d'élèves en CE2 hors ULIS", "Nombre d'élèves en CM1 hors ULIS", "Nombre d'élèves en CM2 hors ULIS", 'num_ligne']


In [8]:
eleves = eleves[eleves["Rentrée scolaire"] == 2024].copy()
eleves["code_departement"] = eleves["Code département"].apply(normalize_code_departement)

numeric_cols = [
    "Nombre total de classes", "Nombre total d'élèves",
    "Nombre d'élèves en pré-élémentaire hors ULIS",
    "Nombre d'élèves en élémentaire hors ULIS",
    "Nombre d'élèves en ULIS",
]
for col in numeric_cols:
    eleves[col] = pd.to_numeric(eleves[col], errors="coerce")

eleves["REP"] = eleves["REP"].astype(str).str.strip().map({"1": "REP", "1.0": "REP"}).fillna("Non REP")
eleves["REP +"] = eleves["REP +"].astype(str).str.strip().map({"1": "REP+", "1.0": "REP+"}).fillna("Non REP+")

eleves_clean = eleves[[
    "Numéro de l'école", "Patronyme",
    "Académie", "Département", "code_departement",
    "Code Postal", "Commune", "Secteur", "REP", "REP +",
    "Nombre total de classes", "Nombre total d'élèves",
]].copy()

eleves_clean.rename(columns={
    "Numéro de l'école": "uai",
    "Patronyme": "nom_ecole",
    "REP +": "REP+",
    "Nombre total de classes": "nb_classes",
    "Nombre total d'élèves": "nb_eleves",
}, inplace=True)

print(f"\nÉcoles nettoyées: {eleves_clean.shape}")
print(f"  Écoles en REP: {(eleves_clean['REP'] == 'REP').sum():,}")
print(f"  Écoles en REP+: {(eleves_clean['REP+'] == 'REP+').sum():,}")
save_processed(eleves_clean, "effectifs_eleves")


Écoles nettoyées: (47413, 12)
  Écoles en REP: 4,131
  Écoles en REP+: 2,458
  -> C:\Users\Salty\Downloads\defi-municipales-2026-enjeux\data\processed\effectifs_eleves.parquet (47,413 rows, 12 cols)


WindowsPath('C:/Users/Salty/Downloads/defi-municipales-2026-enjeux/data/processed/effectifs_eleves.parquet')

## 2.5 Personnels 1er degré

In [9]:
pers = load_csv(DATA_RAW / "fr-en-indicateurs_personnels_etablissements1d.csv", sep=";")
print(f"Personnels 1er degré: {pers.shape}")
print(f"  Colonnes: {pers.columns.tolist()}")

Personnels 1er degré: (47507, 21)
  Colonnes: ['Année de la rentrée scolaire', 'Secteur', "Identifiant de l'établissement", "Nom de l'établissement", 'Code région ref', 'Libellé Région', 'Code académie', 'Libellé académie', 'Code département', 'Libellé departement', "ETP d'enseignants (hommes et femmes)", 'ETP de femmes enseignantes', "ETP d'enseignants de moins de 35 ans", "ETP d'enseignants de 35 à moins de 50 ans", "ETP d'enseignants de 50 ans ou plus", "ETP d'enseignants ayant une ancienneté dans l'établissement de moins de 2 ans", "ETP d'enseignants ayant une ancienneté dans l'établissement de 2 ans à moins de 5 ans", "ETP d'enseignants ayant une ancienneté dans l'établissement de 5 ans à moins de 8 ans", "ETP d'enseignants ayant une ancienneté dans l'établissement de 8 ans ou plus", 'Géolocalisation', 'num_ligne']


In [10]:
pers["code_departement"] = pers["Code département"].apply(normalize_code_departement)

for col in pers.columns:
    if "ETP" in col or "Année" in col:
        pers[col] = pd.to_numeric(pers[col], errors="coerce")

pers_clean = pers[[
    "Identifiant de l'établissement", "Nom de l'établissement",
    "Libellé académie", "Libellé departement", "code_departement",
    "Secteur",
    "ETP d'enseignants (hommes et femmes)",
    "ETP de femmes enseignantes",
    "ETP d'enseignants de moins de 35 ans",
    "ETP d'enseignants de 35 à moins de 50 ans",
    "ETP d'enseignants de 50 ans ou plus",
]].copy()

pers_clean.rename(columns={
    "Identifiant de l'établissement": "uai",
    "Nom de l'établissement": "nom_ecole",
    "ETP d'enseignants (hommes et femmes)": "etp_enseignants",
    "ETP de femmes enseignantes": "etp_femmes",
    "ETP d'enseignants de moins de 35 ans": "etp_moins35",
    "ETP d'enseignants de 35 à moins de 50 ans": "etp_35_50",
    "ETP d'enseignants de 50 ans ou plus": "etp_plus50",
}, inplace=True)

print(f"\nPersonnels nettoyé: {pers_clean.shape}")
save_processed(pers_clean, "personnels_1er_degre")


Personnels nettoyé: (47507, 11)


  -> C:\Users\Salty\Downloads\defi-municipales-2026-enjeux\data\processed\personnels_1er_degre.parquet (47,507 rows, 11 cols)


WindowsPath('C:/Users/Salty/Downloads/defi-municipales-2026-enjeux/data/processed/personnels_1er_degre.parquet')

## 2.6 FiloSoFi - Revenus & Pauvreté

In [11]:
filosofi_zip = DATA_RAW / "base-filosofi-communes-2021.zip"

if filosofi_zip.exists():
    with zipfile.ZipFile(filosofi_zip) as z:
        with z.open("FILO2021_DISP_COM.csv") as f:
            filo_disp = pd.read_csv(f, sep=";", low_memory=False)
        with z.open("FILO2021_DISP_PAUVRES_COM.csv") as f:
            filo_pauv = pd.read_csv(f, sep=";", low_memory=False)
        
        filo = pd.merge(filo_disp, filo_pauv, on="CODGEO", how="outer")
        print(f"Loaded merged FiloSoFi data: {filo.shape}")
else:
    print("Archive FiloSoFi manquante - section ignorée")
    filo = None

Loaded merged FiloSoFi data: (34929, 872)


In [12]:
if filo is not None:
    print(f"FiloSoFi: {filo.shape}")
    print(f"  Colonnes: {filo.columns.tolist()}")
    filo.head(3)
else:
    print("FiloSoFi non disponible")

FiloSoFi: (34929, 872)
  Colonnes: ['CODGEO', 'NBMEN21', 'NBPERS21', 'NBUC21', 'Q121', 'Q221', 'Q321', 'Q3_Q1', 'D121', 'D221', 'D321', 'D421', 'D621', 'D721', 'D821', 'D921', 'RD', 'S80S2021', 'GI21', 'PACT21', 'PTSA21', 'PCHO21', 'PBEN21', 'PPEN21', 'PPAT21', 'PPSOC21', 'PPFAM21', 'PPMINI21', 'PPLOGT21', 'PIMPOT21', 'AGE1Q121', 'AGE1Q221', 'AGE1Q321', 'AGE1Q3_Q1', 'AGE1D121', 'AGE1D221', 'AGE1D321', 'AGE1D421', 'AGE1D621', 'AGE1D721', 'AGE1D821', 'AGE1D921', 'AGE1RD', 'AGE1S80S2021', 'AGE1GI21', 'AGE1PACT21', 'AGE1PTSA21', 'AGE1PCHO21', 'AGE1PBEN21', 'AGE1PPEN21', 'AGE1PPAT21', 'AGE1PPSOC21', 'AGE1PPFAM21', 'AGE1PPMINI21', 'AGE1PPLOGT21', 'AGE1PIMPOT21', 'AGE2Q121', 'AGE2Q221', 'AGE2Q321', 'AGE2Q3_Q1', 'AGE2D121', 'AGE2D221', 'AGE2D321', 'AGE2D421', 'AGE2D621', 'AGE2D721', 'AGE2D821', 'AGE2D921', 'AGE2RD', 'AGE2S80S2021', 'AGE2GI21', 'AGE2PACT21', 'AGE2PTSA21', 'AGE2PCHO21', 'AGE2PBEN21', 'AGE2PPEN21', 'AGE2PPAT21', 'AGE2PPSOC21', 'AGE2PPFAM21', 'AGE2PPMINI21', 'AGE2PPLOGT21', 'AGE2P

In [13]:
if filo is not None:
    # 1. Derive department code
    def get_dep(code):
        if pd.isna(code):
            return code
        code_str = str(code).strip()
        if code_str.startswith("97"):
            return code_str[:3]
        return code_str[:2]
    filo["code_departement"] = filo["CODGEO"].apply(get_dep).apply(normalize_code_departement)
    
    # 2. Derive COM code
    def get_com(code):
        if pd.isna(code):
            return code
        code_str = str(code).strip()
        if code_str.startswith("97"):
            return code_str[3:]
        return code_str[2:]
    filo["COM"] = filo["CODGEO"].apply(get_com)
    
    # 3. Derive commune name (LIBGEO) by merging with Loyers & SRU datasets
    insee_to_name = {}
    sru_path = DATA_PROCESSED / "sru.parquet"
    if sru_path.exists():
        try:
            sru_df = pd.read_parquet(sru_path)
            for _, row in sru_df.iterrows():
                insee_to_name[row["code_insee"]] = row["Nom_commune"]
        except Exception as e:
            print(f"Warning reading SRU for mapping: {e}")
            
    loyers_path = DATA_PROCESSED / "loyers.parquet"
    if loyers_path.exists():
        try:
            loyers_df = pd.read_parquet(loyers_path)
            for _, row in loyers_df.iterrows():
                if row["code_insee"] not in insee_to_name:
                    insee_to_name[row["code_insee"]] = row["LIBGEO"]
        except Exception as e:
            print(f"Warning reading Loyers for mapping: {e}")
            
    filo["LIBGEO"] = filo["CODGEO"].map(insee_to_name)
    filo["LIBGEO"] = filo["LIBGEO"].fillna(filo["CODGEO"])
    
    # 4. Map the indicator columns Q221 (mediane) and TP6021 (pauvreté)
    filo["Médiane du niveau de vie (€)"] = clean_pct(filo["Q221"])
    filo["Taux de pauvreté (%)"] = clean_pct(filo["TP6021"])
    
    key_cols = ["CODGEO", "LIBGEO", "code_departement", "COM"]
    indicator_cols = ["Médiane du niveau de vie (€)", "Taux de pauvreté (%)"]
    
    print(f"Colonnes indicateurs trouvées ({len(indicator_cols)}):")
    for c in indicator_cols:
        print(f"  {c}")
        
    filo_clean = filo[key_cols + indicator_cols].copy()
    filo_clean.rename(columns={"CODGEO": "code_insee", "LIBGEO": "nom_commune"}, inplace=True)
    
    print(f"\nFiloSoFi nettoyé: {filo_clean.shape}")
    save_processed(filo_clean, "filosofi")
else:
    filo_clean = None
    indicator_cols = []
    print("FiloSoFi non disponible - nettoyage ignoré")

Colonnes indicateurs trouvées (2):
  Médiane du niveau de vie (€)
  Taux de pauvreté (%)

FiloSoFi nettoyé: (34929, 6)
  -> C:\Users\Salty\Downloads\defi-municipales-2026-enjeux\data\processed\filosofi.parquet (34,929 rows, 6 cols)


## 2.7 Agrégation départementale

In [14]:
sru_dep = sru_clean.groupby("code_departement").agg(
    nb_communes_sru=("Commune_sru_au_01_01_2025", "sum"),
    nb_communes_deficitaires=("commune_deficitaire", "sum"),
    nb_communes_carencees=("Commune_carencée", "sum"),
    taux_sru_moyen=("Taux_SRU_au_01_01_2024", "mean"),
    nb_logements_sociaux=("Nombre_lls_ Inventaire_au_01_01_2024", "sum"),
    population_totale=("Population_municipale_01_01_2025", "sum"),
).reset_index()

loyer_dep = loyers[loyers["type_logement"] == "appartement"].groupby("code_departement").agg(
    loyer_moyen_m2=("loypredm2", "mean"),
    nb_communes_loyer=("code_insee", "nunique"),
).reset_index()

eleves_dep = eleves_clean.groupby("code_departement").agg(
    nb_ecoles=("uai", "nunique"),
    nb_eleves_total=("nb_eleves", "sum"),
    nb_classes_total=("nb_classes", "sum"),
    nb_ecoles_rep=("REP", lambda x: (x == "REP").sum()),
    nb_ecoles_repp=("REP+", lambda x: (x == "REP+").sum()),
).reset_index()
eleves_dep["ratio_eleves_classe"] = (
    eleves_dep["nb_eleves_total"] / eleves_dep["nb_classes_total"]
)
eleves_dep["pct_ecoles_rep"] = (
    (eleves_dep["nb_ecoles_rep"] + eleves_dep["nb_ecoles_repp"]) / eleves_dep["nb_ecoles"] * 100
)

pers_dep = pers_clean.groupby("code_departement").agg(
    etp_enseignants_total=("etp_enseignants", "sum"),
).reset_index()

edu_dep = eleves_dep.merge(pers_dep, on="code_departement", how="left")
edu_dep["ratio_eleves_enseignant"] = (
    edu_dep["nb_eleves_total"] / edu_dep["etp_enseignants_total"]
)

print(f"SRU par département: {sru_dep.shape}")
print(f"Loyers par département: {loyer_dep.shape}")
print(f"Éducation par département: {edu_dep.shape}")
save_processed(sru_dep, "sru_par_departement")
save_processed(loyer_dep, "loyers_par_departement")
save_processed(edu_dep, "education_par_departement")

SRU par département: (94, 7)
Loyers par département: (100, 3)
Éducation par département: (101, 10)
  -> C:\Users\Salty\Downloads\defi-municipales-2026-enjeux\data\processed\sru_par_departement.parquet (94 rows, 7 cols)
  -> C:\Users\Salty\Downloads\defi-municipales-2026-enjeux\data\processed\loyers_par_departement.parquet (100 rows, 3 cols)
  -> C:\Users\Salty\Downloads\defi-municipales-2026-enjeux\data\processed\education_par_departement.parquet (101 rows, 10 cols)


WindowsPath('C:/Users/Salty/Downloads/defi-municipales-2026-enjeux/data/processed/education_par_departement.parquet')

In [15]:
if filo_clean is not None:
    filo_dep_cols = ["code_departement"] + [c for c in indicator_cols if c in filo_clean.columns]
    filo_dep = filo_clean[filo_dep_cols].groupby("code_departement").mean(numeric_only=True).reset_index()
    print(f"FiloSoFi par département: {filo_dep.shape}")
    print(f"  Colonnes: {filo_dep.columns.tolist()}")
    save_processed(filo_dep, "filosofi_par_departement")

FiloSoFi par département: (98, 3)
  Colonnes: ['code_departement', 'Médiane du niveau de vie (€)', 'Taux de pauvreté (%)']
  -> C:\Users\Salty\Downloads\defi-municipales-2026-enjeux\data\processed\filosofi_par_departement.parquet (98 rows, 3 cols)


## 2.8 Vue d'ensemble des fichiers traités

In [16]:
print("=== Fichiers traités ===")
for f in sorted(DATA_PROCESSED.iterdir()):
    if f.suffix == ".parquet":
        df = pd.read_parquet(f)
        print(f"  {f.name:45s} {df.shape[0]:>8,} rows x {df.shape[1]:>3} cols")

=== Fichiers traités ===
  education_par_departement.parquet                  101 rows x  10 cols
  education_prioritaire.parquet                    8,501 rows x  12 cols
  effectifs_eleves.parquet                        47,413 rows x  12 cols
  filosofi.parquet                                34,929 rows x   6 cols
  filosofi_par_departement.parquet                    98 rows x   3 cols
  loyers.parquet                                 139,600 rows x   6 cols
  loyers_par_departement.parquet                     100 rows x   3 cols
  personnels_1er_degre.parquet                    47,507 rows x  11 cols
  sru.parquet                                      2,196 rows x  13 cols
  sru_par_departement.parquet                         94 rows x   7 cols
